In [ ]:
!pip install numba numpy

In [ ]:
from numba import cuda
import numpy as np
import math
import time

In [ ]:
#cuda kernel
@cuda.jit
def add_vector(A,B,C):
    idx = cuda.grid(1)

    if idx < A.size:
        C[idx] = A[idx] + B[idx]

In [ ]:
# size of vector
N = 100000

# input vector
A = np.random.randint(0,100, N).astype(np.float32)
B = np.random.randint(0,100, N).astype(np.float32)

# output vector
C = np.zeros(N,dtype=np.float32)


#copy to GPU

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.device_array_like(C)


# Threads config
threads_per_block =256
blocks_per_grid = math.ceil(N/threads_per_block)

In [ ]:
# CPU Timing
start_cpu = time.perf_counter()

for i in range(N):
    C[i] = A[i] + B[i]

end_cpu = time.perf_counter()

In [ ]:
# GPUUU
# Start timing
start_gpu = time.perf_counter()

# launch kernel
add_vector[blocks_per_grid, threads_per_block](d_A,d_B,d_C)

# Wait for GPU to finish
cuda.synchronize()

# copy result back
C = d_C.copy_to_host()

# End timing
end_gpu = time.perf_counter()

In [ ]:
print("A[:5] = ", A[:5])
print("B[:5] = ", B[:5])
print("C[:5] = ", C[:5])

print("\nGPU Execution Time:", (end_gpu - start_gpu) * 1000, "ms")
print("\nCPU Execution Time:", (end_cpu - start_cpu) * 1000, "ms")

A[:5] =  [96. 96. 73. 28. 22.]
B[:5] =  [85. 79. 37. 50. 25.]
C[:5] =  [181. 175. 110.  78.  47.]

GPU Execution Time: 1.3197450000461686 ms

CPU Execution Time: 41.0912099999905 ms
